# AutoRAG: базовый шаблон RAG-пайплайна

Этот ноутбук демонстрирует **минимальный пример** использования AutoRAG для задач Retrieval-Augmented Generation (RAG).

Мы по шагам:
1. Установим и импортируем AutoRAG.
2. Настроим пути к данным (QA и корпус документов).
3. Сформируем конфиг для AutoRAG **прямо в ноутбуке** (как строку YAML), с подробными комментариями.
4. Запишем конфиг в файл и запустим `Evaluator.start_trial`.
5. Быстро посмотрим результаты эксперимента.
6. Покажем пример использования найденного лучшего пайплайна через `Runner`.

⚠️ **Важно:** ноутбук — шаблон. Тебе нужно будет подставить свои пути к данным и свой API‑ключ LLM (если используется внешний сервис).

In [ ]:
# Шаг 1. Установка AutoRAG и базовые импорты

# Если AutoRAG ещё не установлен в текущем окружении,
# раскомментируй строку ниже и выполни ячейку один раз.
# Вариант с поддержкой GPU и парсинга документов:
# !pip install "AutoRAG[gpu,parse]"

# Импортируем стандартные библиотеки
import os  # для работы с переменными окружения и путями
from pathlib import Path  # удобная работа с путями

# Импортируем основные компоненты AutoRAG
from autorag.evaluator import Evaluator  # класс для запуска экспериментов и оптимизации RAG
from autorag.deploy import Runner        # класс для загрузки и использования лучшего пайплайна

print("Импорты завершены.")

In [ ]:
# Шаг 2. Настройка путей к данным и переменных окружения

# Определяем корневую директорию проекта — здесь находится этот ноутбук (.ipynb).
PROJECT_ROOT = Path.cwd()

# Папка с данными. Предполагается, что внутри будут файлы:
#  - qa.parquet    — датасет вопрос-ответ (QA) для обучения/оценки
#  - corpus.parquet — корпус документов, по которому будет работать retrieval
DATA_DIR = PROJECT_ROOT / "data"

# Папка для конфигов (мы будем записывать туда YAML-конфиг AutoRAG).
CONFIG_DIR = PROJECT_ROOT / "config"

# Папка для результатов экспериментов AutoRAG (логов, summary и т.д.).
BENCHMARK_DIR = PROJECT_ROOT / "benchmark"

# Гарантируем, что папки под конфиг и результаты существуют (если их нет — создадим).
CONFIG_DIR.mkdir(exist_ok=True)
BENCHMARK_DIR.mkdir(exist_ok=True)

# Полные пути к файлам с данными.
QA_PATH = DATA_DIR / "qa.parquet"         # путь к QA-датасету
CORPUS_PATH = DATA_DIR / "corpus.parquet" # путь к корпусу документов

# Имя файла с конфигом AutoRAG. Он будет создан автоматически из строки в следующей ячейке.
CONFIG_PATH = CONFIG_DIR / "config_autorag.yaml"

# Если используешь внешнюю LLM (например, OpenAI-совместимый API),
# здесь можно указать API-ключ. Для безопасности лучше задавать его
# вне ноутбука (в переменных окружения системы) и не хранить в коде.
# Ниже оставлен пример строки, которую НУЖНО заменить или удалить:
# os.environ["OPENAI_API_KEY"] = "sk-..."  # ⚠️ ЗАМЕНИ или убери эту строку

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Папка с данными:", DATA_DIR)
print("Файл QA:", QA_PATH)
print("Файл корпуса:", CORPUS_PATH)
print("Файл конфига:", CONFIG_PATH)
print("Папка для бенчмарков:", BENCHMARK_DIR)

## Ожидаемый формат данных

AutoRAG ожидает, что:

- **`qa.parquet`** — таблица с примерами вида *вопрос → ответ*.
  Примеры типичных колонок:
  - `question` — текст вопроса пользователя;
  - `answer` — эталонный ответ (ground truth), который нужен для вычисления метрик;
  - при необходимости могут быть и другие поля, зависящие от конкретного пайплайна.

- **`corpus.parquet`** — таблица с документами для поиска.
  Примеры колонок:
  - `doc_id` — уникальный идентификатор документа;
  - `content` — текст документа (параграф / чанк / статья);
  - дополнительные поля с метаданными (заголовок, источник и т.д.) по желанию.

В этом шаблоне мы **не создаём данные**, а предполагаем, что они уже подготовлены.
Если их нет — нужно будет предварительно спарсить свои документы и собрать QA-датасет.

In [ ]:
# Шаг 3. Определяем конфиг AutoRAG как строку YAML (внутри Python-строки)

# ВАЖНО: здесь мы описываем конфигурацию целиком внутри ноутбука.
# Это обычная многострочная строка Python с YAML-содержимым.
# Далее мы запишем её в файл CONFIG_PATH, так как AutoRAG ожидает путь к YAML-файлу.

config_text = r"""
# ==========================
# AutoRAG config (шаблон)
# ==========================
# Этот YAML-файл описывает, какие комбинации узлов (nodes) AutoRAG
# будет перебирать при построении и оценке RAG-пайплайна.
#
# Основные сущности:
# - node_lines: список "линий" узлов, соответствующих этапам пайплайна.
# - nodes: конкретные узлы внутри каждой линии (retriever, prompt_maker, generator и т.д.).
# - modules: конкретные реализации внутри узлов (bm25, vectordb, openai_llm и др.).
#
# В этом базовом шаблоне есть две линии:
# 1) retrieve_node_line       — за извлечение (retrieval) релевантных документов.
# 2) post_retrieve_node_line  — за формирование промпта и генерацию ответа LLM.

node_lines:
  # ----------------------------
  # Линия 1: Retrieval (извлечение документов)
  # ----------------------------
  - node_line_name: retrieve_node_line
    nodes:
      # Узел 1. Лексический ретривер (BM25)
      - node_type: lexical_retrieval
        strategy:
          # Метрики качества извлечения, по которым будет оцениваться узел.
          metrics: [retrieval_f1, retrieval_recall, retrieval_ndcg, retrieval_mrr]
        top_k: 3  # сколько документов извлекать для каждого запроса
        modules:
          - module_type: bm25  # конкретная реализация лексического поиска

      # Узел 2. Семантический ретривер (векторная БД)
      - node_type: semantic_retrieval
        strategy:
          metrics: [retrieval_f1, retrieval_recall, retrieval_ndcg, retrieval_mrr]
        top_k: 3
        modules:
          - module_type: vectordb   # модуль векторной БД
            vectordb: default       # имя подключения к векторной БД (создаётся при индексации)

      # Узел 3. Гибридный ретривер (lexical + semantic)
      - node_type: hybrid_retrieval
        strategy:
          metrics: [retrieval_f1, retrieval_recall, retrieval_ndcg, retrieval_mrr]
        top_k: 3
        modules:
          - module_type: hybrid_rrf   # RRF (Reciprocal Rank Fusion) для объединения выдачи
            weight_range: (4, 80)     # диапазон весов, который будет перебирать AutoRAG

  # ----------------------------
  # Линия 2: Post-retrieval (формирование промпта и генерация)
  # ----------------------------
  - node_line_name: post_retrieve_node_line
    nodes:
      # Узел 4. Prompt Maker — формирует итоговый текстовый промпт для LLM
      - node_type: prompt_maker
        strategy:
          # Метрики качества генерации (по сравнению с эталонным ответом в QA-датасете).
          metrics:
            - metric_name: meteor
            - metric_name: rouge
            - metric_name: sem_score
              embedding_model: openai  # какую модель эмбеддингов использовать для семантической оценки
        modules:
          - module_type: fstring
            # Шаблон промпта; {query} и {retrieved_contents} подставляются AutoRAG.
            prompt: |
              Read the passages and answer the given question.
              Question: {query}
              Passages: {retrieved_contents}
              Answer:

      # Узел 5. Generator — запускает LLM для генерации ответа
      - node_type: generator
        strategy:
          metrics:
            - metric_name: meteor
            - metric_name: rouge
            - metric_name: sem_score
              embedding_model: openai
        modules:
          - module_type: openai_llm
            # Базовая конфигурация LLM.
            # Здесь указана одна модель и batch-размер.
            # При желании можно добавить несколько моделей или параметров
            # и AutoRAG будет перебирать разные комбинации.
            llm: gpt-4o-mini
            batch: 16
"""

print("Конфиг AutoRAG определён как Python-строка (YAML).")

In [ ]:
# Шаг 4. Сохраняем YAML-конфиг в файл, который будет читать AutoRAG

# Записываем строку config_text в файл CONFIG_PATH.
# Это нужно, потому что Evaluator.start_trial ожидает путь к YAML-файлу.

CONFIG_PATH.write_text(config_text, encoding="utf-8")
print(f"Конфиг сохранён в файл: {CONFIG_PATH}")

In [ ]:
# Шаг 5. Запуск эксперимента AutoRAG через Evaluator

# Перед запуском проверим, что файлы с данными действительно существуют.
if not QA_PATH.exists():
    raise FileNotFoundError(f"QA файл не найден: {QA_PATH}. Подготовь qa.parquet перед запуском.")
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f"Файл корпуса не найден: {CORPUS_PATH}. Подготовь corpus.parquet перед запуском.")

# Создаём экземпляр Evaluator. В него передаём пути к QA-датасету и корпусу,
# а также папку, где будут храниться результаты бенчмарка и триалов.
evaluator = Evaluator(
    qa_data_path=str(QA_PATH),
    corpus_data_path=str(CORPUS_PATH),
    project_dir=str(BENCHMARK_DIR),
)

# Запускаем триал с нашим конфигом.
# AutoRAG прочитает CONFIG_PATH, переберёт заданные комбинации модулей
# и рассчитает метрики, сохранив результаты в папку BENCHMARK_DIR.

print("Запускаем Evaluator.start_trial...")
evaluator.start_trial(str(CONFIG_PATH))
print("Триал завершён. Результаты сохранены в папке:", BENCHMARK_DIR)

In [ ]:
# Шаг 6. Быстрый просмотр результатов эксперимента (summary.csv)

import pandas as pd  # импортируем pandas для удобной работы с таблицами

# AutoRAG создаёт папки триалов с порядковыми номерами: 0, 1, 2, ...
# В демонстрации предполагаем, что запуск был первым, и папка — "0".
trial_dir = BENCHMARK_DIR / "0"
summary_path = trial_dir / "summary.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Файл summary.csv не найден по пути: {summary_path}. "
        "Убедись, что триал успешно отработал и путь задан верно."
    )

# Читаем summary.csv в DataFrame.
summary_df = pd.read_csv(summary_path)

# Выводим первые строки, чтобы увидеть список комбинаций модулей и метрик.
print("Первые строки summary.csv:")
display(summary_df.head())

# При желании можно отсортировать по нужной метрике, например по sem_score или rouge.
# Ниже пример сортировки по колонке 'sem_score' (если она есть в summary).
if 'sem_score' in summary_df.columns:
    print("\nТоп-5 конфигураций по sem_score:")
    display(summary_df.sort_values(by='sem_score', ascending=False).head())
else:
    print("\nКолонка 'sem_score' не найдена в summary.csv. Посмотри доступные колонки:")
    print(summary_df.columns.tolist())

In [ ]:
# Шаг 7. Использование лучшего найденного пайплайна через Runner

# Runner умеет загружать конфигурацию и веса (если есть) лучшей комбинации узлов,
# найденной в рамках триала, и предоставлять удобный метод run(question).

# Загружаем Runner из папки триала (здесь снова используем триал с номером "0").
trial_dir = BENCHMARK_DIR / "0"
runner = Runner.from_trial_folder(str(trial_dir))

# Пример вопроса, на который мы хотим получить ответ с использованием оптимального RAG-пайплайна.
demo_question = "What is the main topic of our knowledge base?"  # пример; замени на свой вопрос

print("Вопрос к пайплайну:", demo_question)

# Запускаем пайплайн: внутри произойдёт retrieval, формирование промпта и генерация ответа.
demo_answer = runner.run(demo_question)

print("\nОтвет пайплайна:")
print(demo_answer)